# 03 - Model Training
Train the tracking error model with temporal validation.

In [2]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# Ensure project modules are importable when running from notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Ensure notebook kernel has required dependencies for data + model modules.
for package in ["yfinance", "shap"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from config import PAIR_CONFIGS
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer
from src.models import TrackingErrorModel

loader = MarketDataLoader()
panel = loader.fetch_universe(PAIR_CONFIGS, period='2y', interval='1d')
features = FeatureEngineer(rolling_window=20, horizon=1).transform_universe(panel)
model = TrackingErrorModel(random_state=42)
result = model.train(features, target_col='target_te', test_size=0.2)
result

c:\Users\Alexandre\anaconda3\envs\catan\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TrainResult(metrics={'mae': 0.0002630855561563704, 'rmse': 0.0007956269309198496, 'r2': 0.9435327012365649, 'mape': 0.08296606201414168}, train_rows=1396, test_rows=350)

In [4]:
artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "te_model.joblib"
model.save(model_path)
str(model_path)

'C:\\Users\\Alexandre\\Documents\\code\\ETF-error-tracking\\artifacts\\te_model.joblib'